# Topic-level data exploration

This notebook code:
1) Loads all files from data/subject/raw
2) Groups them by logical topic (Topic_01, Topic_02, etc.)
3) Builds one "topic document" by concatenating all files for that topic
4) Computes basic statistics (words, characters, number of files)
5) Builds a simple vocabulary with normalised tokens (lowercase, no stopwords)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install PyPDF2 python-docx python-pptx docx2txt nltk scikit-learn pandas matplotlib

!apt-get -y update
!apt-get -y install libreoffice

In [ ]:
import os
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from PyPDF2 import PdfReader
from pptx import Presentation
import docx2txt
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
import subprocess
import tempfile

import nltk
from nltk.stem import WordNetLemmatizer


# Configuration

In [ ]:
# Root folder with data/<subject>/raw/.
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/data")

# Subjects you want to analyse
SUBJECTS = ["fsd", "fcs", "dma"]

# Output directory for CSVs and figures
OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/analysis_topic_level")
FIGS_DIR = OUTPUT_DIR / "figs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# Match filenames like:
#   Topic_01_DMA Week 1 Tutorial
#   Topic_10_Whatever
TOPIC_PATTERN = re.compile(r"^Topic_(\d+)", re.IGNORECASE)

# Stemmer for normalising word forms (running -> run, graphs -> graph, etc.)
STEMMER = PorterStemmer()

# Download WordNet data once
nltk.download("wordnet")

DOMAIN_STOPWORDS = {
    # links/domains
    "http", "https", "www", "com", "org", "net",
    "youtube", "watch", "wikipedia", "wiki", "en",

    # material format
    "tutorial", "lecture", "note", "answer", "sheet", "week",

    # frequent PDF/extraction artifacts
    "ofa", "intmain", "bn", "pr", "nm"
}

STOP_WORDS = set(ENGLISH_STOP_WORDS) | DOMAIN_STOPWORDS

LEMMATIZER = WordNetLemmatizer()

URL_EMAIL_PATTERN = re.compile(
    r"(https?://\S+|www\.\S+|\b\S+@\S+\.\S+\b)",
    flags=re.IGNORECASE
)

# Token pattern: only alphabetic tokens, at least 2 characters
TOKEN_PATTERN = re.compile(r"[A-Za-z]{2,}")


# Loading files

In [ ]:
def convert_doc_to_docx(doc_path: Path) -> Path:
    """
    Convert .doc -> .docx using LibreOffice headless.
    Returns path to the converted .docx (in a temp directory).
    """
    doc_path = Path(doc_path)
    if doc_path.suffix.lower() != ".doc":
        raise ValueError(f"Expected .doc file, got {doc_path}")

    tmpdir = Path(tempfile.mkdtemp(prefix="doc2docx_"))
    # LibreOffice writes output into outdir with same stem
    cmd = [
        "libreoffice",
        "--headless",
        "--convert-to", "docx",
        str(doc_path),
        "--outdir", str(tmpdir),
    ]

    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except FileNotFoundError:
        raise RuntimeError("LibreOffice not found. Install it (apt-get install libreoffice) or add it to PATH.")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"LibreOffice conversion failed for {doc_path}:\n{e.stderr}") from e

    out_path = tmpdir / f"{doc_path.stem}.docx"
    if not out_path.exists():
        # Sometimes LO changes naming, but usually it doesn't. Fail loudly if not found.
        raise RuntimeError(f"Conversion succeeded but output not found: {out_path}")

    return out_path


In [ ]:
def load_pdf_text(path: Path) -> str:
    """Extract all text from a PDF file (all pages concatenated)."""
    try:
        reader = PdfReader(str(path))
        texts = []
        for page in reader.pages:
            try:
                page_text = page.extract_text() or ""
            except Exception:
                page_text = ""
            if page_text:
                texts.append(page_text)
        return "\n".join(texts)
    except Exception as e:
        print(f"[ERROR] Failed to read PDF {path.name}: {e}")
        return ""


def load_pptx_text(path: Path) -> str:
    """Extract all text from a PPTX file (all slides)."""
    try:
        prs = Presentation(str(path))
        texts = []
        for slide in prs.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text:
                    texts.append(shape.text)
        return "\n".join(texts)
    except Exception as e:
        print(f"[ERROR] Failed to read PPTX {path.name}: {e}")
        return ""


def load_docx_text(path: Path) -> str:
    """Extract text from a DOCX file using docx2txt."""
    try:
        text = docx2txt.process(str(path))
        return text or ""
    except Exception as e:
        print(f"[ERROR] Failed to read DOCX {path.name}: {e}")
        return ""


def load_txt_md_text(path: Path) -> str:
    """Load plain text or markdown file."""
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"[ERROR] Failed to read text file {path.name}: {e}")
        return ""


def load_file_text(path: Path) -> str:
    """Dispatch to the correct loader based on file extension."""
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf_text(path)
    elif suffix == ".pptx":
        return load_pptx_text(path)
    elif suffix == ".docx":
        return load_docx_text(path)
    elif suffix == ".doc":
        converted = convert_doc_to_docx(path)
        return docx2txt.process(str(converted))
    elif suffix in {".txt", ".md"}:
        return load_txt_md_text(path)
    else:
        print(f"[INFO] Skipping unsupported file type: {path.name}")
        return ""

# Topic extraction

In [ ]:
def extract_topic_id(file_name: str, subject: str) -> str:
    """
    Extract a topic ID from the filename.

    Expected format now:
        Topic_01_DMA Week 1 Tutorial Answers.docx
        Topic_10_Discrete Optimization.pdf

    The topic key will be normalised as:
        <SUBJECT>_Topic_01, <SUBJECT>_Topic_10, etc.

    If the pattern does not match, we fall back to:
        <SUBJECT>_MISC_<stem>
        so that the file is still included in the analysis.
    """
    stem = Path(file_name).stem
    m = TOPIC_PATTERN.match(stem)
    if m:
        num = int(m.group(1))
        topic_num = f"{num:02d}"
        subject_upper = subject.upper()
        return f"{subject_upper}_Topic_{topic_num}"

    # Fallback: if filename doesn't follow the pattern, still keep it
    subject_upper = subject.upper()
    print(
        f"[WARNING] Could not parse topic from '{file_name}'. "
        f"Assigning fallback topic ID based on filename stem."
    )
    return f"{subject_upper}_MISC_{stem}"


# NLP preprocessing

In [ ]:
def tokenize_normalize(text: str):
    """
    Tokenise and normalise text:
    - lowercase
    - keep only alphabetic tokens with length >= 2
    - remove English stop words
    - lemmatise remaining tokens
    """
    # Pre-clean + Lowercase
    text = URL_EMAIL_PATTERN.sub(" ", text.lower())

    # Basic tokenisation by regex: only alphabetic tokens
    tokens = TOKEN_PATTERN.findall(text)

    # Remove stop words
    tokens = [t for t in tokens if t not in STOP_WORDS]

    # Lemmatise tokens (map different forms to a common lemma)
    lemmas = [LEMMATIZER.lemmatize(t) for t in tokens]

    # Сollapse consecutive duplicates (noise from extraction)
    dedup = []
    for t in lemmas:
        if not dedup or dedup[-1] != t:
            dedup.append(t)
    return dedup


def generate_ngrams(tokens, n=2):
    """Return list of n-grams (as tuples) from a token list."""
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


# Build topic-level corpus

In [ ]:
def build_topic_corpus():
    """
    Build a topic-level corpus for all subjects.

    One topic = one "document" = all files with the same topic ID
    concatenated into a single text.
    """
    topics = {}  # (subject, topic_id) -> dict with text and file metadata

    for subject in SUBJECTS:
        subject_upper = subject.upper()
        raw_dir = BASE_DIR / subject / "raw"
        if not raw_dir.exists():
            print(f"[WARNING] Raw directory not found for {subject_upper}: {raw_dir}")
            continue

        print(f"\n[INFO] Processing subject: {subject_upper}")
        print(f"[INFO] Raw directory: {raw_dir}")

        for file_path in sorted(raw_dir.glob("*")):
            if file_path.is_dir():
                continue

            text = load_file_text(file_path)
            if not text.strip():
                # Empty or unreadable file
                continue

            # Topic ID from filename
            topic_id = extract_topic_id(file_path.name, subject)

            # Initialize topic record if needed
            key = (subject_upper, topic_id)
            if key not in topics:
                topics[key] = {
                    "subject": subject_upper,
                    "topic_id": topic_id,
                    "text_parts": [],
                    "files": [],  # list of per-file metadata
                }

            # Append text and file metadata
            word_count = len(text.split())
            topics[key]["text_parts"].append(text)
            topics[key]["files"].append(
                {
                    "file_name": file_path.name,
                    "suffix": file_path.suffix.lower(),
                    "words": word_count,
                }
            )

    # Merge text parts into a single string per topic
    for key, data in topics.items():
        data["full_text"] = "\n\n".join(data["text_parts"])
        data["total_words"] = len(data["full_text"].split())
        data["total_chars"] = len(data["full_text"])

    print(f"\n[INFO] Built topic-level corpus with {len(topics)} topics in total.")
    return topics

# Topic-level statistics and save to CSV

In [ ]:
def summarize_topics(topics):
    """
    Create a DataFrame with topic-level statistics and save to CSV.

    Columns include:
        - subject
        - topic_id
        - n_files
        - n_pdf, n_pptx, n_docx, n_txt
        - total_words
        - total_chars
        - mean_words_per_file
    """
    records = []

    for (subject, topic_id), data in topics.items():
        files = data["files"]
        total_words = data["total_words"]
        total_chars = data["total_chars"]

        n_files = len(files)
        type_counts = Counter(f["suffix"] for f in files)

        record = {
            "subject": subject,
            "topic_id": topic_id,
            "n_files": n_files,
            "n_pdf": type_counts.get(".pdf", 0),
            "n_pptx": type_counts.get(".pptx", 0),
            "n_docx": type_counts.get(".docx", 0),
            "n_txt": type_counts.get(".txt", 0) + type_counts.get(".md", 0),
            "total_words": total_words,
            "total_chars": total_chars,
        }
        record["mean_words_per_file"] = total_words / n_files if n_files > 0 else 0.0

        records.append(record)

    topics_df = pd.DataFrame(records)
    topics_df.sort_values(["subject", "topic_id"], inplace=True)

    # Save full topic-level table
    topics_csv_path = OUTPUT_DIR / "topics_summary.csv"
    topics_df.to_csv(topics_csv_path, index=False)
    print(f"[INFO] Saved topic-level summary to {topics_csv_path}")

    return topics_df

# Subject-level statistics

In [ ]:
def summarize_corpus_by_subject(topics_df, subject_token_counter):
    """
    Corpus-level statistics per subject (after preprocessing).

    Returns DataFrame with:
        - subject: subject name
        - topics: number of topics
        - total_tokens: sum of all tokens (after preprocessing)
        - unique_tokens: vocabulary size (after preprocessing)
    """
    records = []

    for subject in sorted(topics_df["subject"].unique()):
        n_topics = len(topics_df[topics_df["subject"] == subject])

        # Both metrics from preprocessed tokens
        token_counter = subject_token_counter.get(subject, {})
        total_tokens = sum(token_counter.values())
        unique_tokens = len(token_counter)

        records.append({
            "subject": subject,
            "topics": n_topics,
            "total_tokens": int(total_tokens),
            "unique_tokens": unique_tokens,
        })

    df = pd.DataFrame(records)
    out_path = OUTPUT_DIR / "corpus_stats_by_subject.csv"
    df.to_csv(out_path, index=False)
    print(f"[INFO] Saved corpus stats to {out_path}")

    return df

# Vocabulary and N-gram analysis (per subject)

In [ ]:
def analyze_vocabulary(
    topics,
    top_k_tokens=50,
    top_k_ngrams=50,
    out_dir=None,
    verbose=True,
):
    subject_token_counter = defaultdict(Counter)
    subject_bigram_counter = defaultdict(Counter)
    subject_trigram_counter = defaultdict(Counter)

    # for progress reporting
    subject_topic_counts = defaultdict(int)

    for (subject, topic_id), data in topics.items():
        subject_topic_counts[subject] += 1

        full_text = data.get("full_text", "")
        tokens = tokenize_normalize(full_text)

        subject_token_counter[subject].update(tokens)
        subject_bigram_counter[subject].update(generate_ngrams(tokens, n=2))
        subject_trigram_counter[subject].update(generate_ngrams(tokens, n=3))

    if verbose:
        for subject in sorted(subject_topic_counts.keys()):
            tok_ctr = subject_token_counter[subject]
            bg_ctr = subject_bigram_counter[subject]
            tg_ctr = subject_trigram_counter[subject]
            print(f"[INFO] Vocabulary summary for {subject}:")
            print(f"  topics processed: {subject_topic_counts[subject]}")
            print(f"  total tokens: {sum(tok_ctr.values())}")
            print(f"  unique tokens: {len(tok_ctr)}")
            print(f"  unique bigrams: {len(bg_ctr)}")
            print(f"  unique trigrams: {len(tg_ctr)}")

    # Build DataFrames (top-K per subject)
    top_tokens_rows = []
    for subject, ctr in subject_token_counter.items():
        for tok, cnt in ctr.most_common(top_k_tokens):
            top_tokens_rows.append({"subject": subject, "token": tok, "count": cnt})
    top_tokens_df = pd.DataFrame(top_tokens_rows)

    top_bigrams_rows = []
    for subject, ctr in subject_bigram_counter.items():
        for bg, cnt in ctr.most_common(top_k_ngrams):
            top_bigrams_rows.append({"subject": subject, "bigram": " ".join(bg), "count": cnt})
    top_bigrams_df = pd.DataFrame(top_bigrams_rows)

    top_trigrams_rows = []
    for subject, ctr in subject_trigram_counter.items():
        for tg, cnt in ctr.most_common(top_k_ngrams):
            top_trigrams_rows.append({"subject": subject, "trigram": " ".join(tg), "count": cnt})
    top_trigrams_df = pd.DataFrame(top_trigrams_rows)

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        top_tokens_path = out_dir / "top_tokens_per_subject.csv"
        top_bigrams_path = out_dir / "top_bigrams_per_subject.csv"
        top_trigrams_path = out_dir / "top_trigrams_per_subject.csv"

        top_tokens_df.to_csv(top_tokens_path, index=False)
        top_bigrams_df.to_csv(top_bigrams_path, index=False)
        top_trigrams_df.to_csv(top_trigrams_path, index=False)

        if verbose:
            print(f"[INFO] Saved: {top_tokens_path}")
            print(f"[INFO] Saved: {top_bigrams_path}")
            print(f"[INFO] Saved: {top_trigrams_path}")

    return (
        subject_token_counter,
        subject_bigram_counter,
        subject_trigram_counter,
        top_tokens_df,
        top_bigrams_df,
        top_trigrams_df,
    )

# Plots

In [ ]:
def plot_length_distribution(lengths, title, outpath=None, bins=50):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(lengths, bins=bins)
    ax.set_title(title)
    ax.set_xlabel("Words per document")
    ax.set_ylabel("Number of documents")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if outpath:
        outpath = Path(outpath)
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200)
    plt.show()

def plot_wordcloud(counter, title, outpath=None, max_words=150):
    if not counter or sum(counter.values()) == 0:
        print(f"[SKIP] Word cloud: no tokens for {title}")
        return

    wc = WordCloud(width=1200, height=600, background_color="white", max_words=max_words)
    wc = wc.generate_from_frequencies(counter)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if outpath:
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200)
    plt.show()

def plot_top_words(counter, title, outpath=None, top_k=20):
    if not counter:
        print(f"[SKIP] Top words: no tokens for {title}")
        return

    top = counter.most_common(top_k)

    words = [w for w, _ in top][::-1]
    freqs = [c for _, c in top][::-1]
    total = sum(counter.values()) or 1

    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(words, freqs)

    ax.set_title(title)
    ax.set_xlabel("Frequency")

    max_f = max(freqs) if freqs else 0
    pad = max_f * 0.01

    for b, f in zip(bars, freqs):
        pct = 100.0 * f / total
        ax.text(
            b.get_width() + pad,
            b.get_y() + b.get_height() / 2,
            f"{f} ({pct:.1f}%)",
            va="center",
            ha="left",
            fontsize=10
        )

    ax.set_xlim(0, max_f * 1.15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if outpath:
        outpath = Path(outpath)
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200)

    plt.show()

def plot_top_ngrams_from_counter(counter, title, outpath=None, top_k=20):
    """
    Bar chart for top-K n-grams stored in Counter with tuple keys, e.g. ('memory','management').
    Adds count and percent labels.
    """
    if not counter or sum(counter.values()) == 0:
        print(f"[SKIP] Top n-grams: no data for {title}")
        return

    top = counter.most_common(top_k)
    labels = [" ".join(ng) for ng, _ in top][::-1]
    freqs  = [c for _, c in top][::-1]

    total = sum(counter.values()) or 1

    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(labels, freqs)

    ax.set_title(title)
    ax.set_xlabel("Frequency")

    max_f = max(freqs) if freqs else 0
    pad = max_f * 0.01

    for b, f in zip(bars, freqs):
        pct = 100.0 * f / total
        ax.text(
            b.get_width() + pad,
            b.get_y() + b.get_height() / 2,
            f"{f} ({pct:.1f}%)",
            va="center",
            ha="left",
            fontsize=10
        )

    ax.set_xlim(0, max_f * 1.15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if outpath:
        outpath = Path(outpath)
        outpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200)

    plt.show()

def plot_vocabulary_size(subject_token_counter, outpath=None):
    """
    Bar chart comparing vocabulary size (unique words) and total tokens per subject.
    Also prints Type-Token Ratio (TTR) for lexical diversity.
    """
    subjects = sorted(subject_token_counter.keys())
    vocab_sizes = [len(subject_token_counter[s]) for s in subjects]
    total_tokens = [sum(subject_token_counter[s].values()) for s in subjects]

    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(subjects))
    width = 0.35

    bars1 = ax.bar(x - width/2, vocab_sizes, width, label="Unique tokens", color="#4C72B0")
    bars2 = ax.bar(x + width/2, total_tokens, width, label="Total tokens", color="#55A868")

    ax.set_xlabel("Subject")
    ax.set_ylabel("Count")
    ax.set_title("Vocabulary Size vs Total Tokens by Subject")
    ax.set_xticks(x)
    ax.set_xticklabels(subjects)
    ax.legend()

    # Add count labels on top of bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if outpath:
        Path(outpath).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200)
    plt.show()

    # Print vocabulary statistics table
    print(chr(10) + "Vocabulary Statistics:")

    for s, v, t in zip(subjects, vocab_sizes, total_tokens):
        ttr = v / t if t > 0 else 0
        print(f"  {s}: {v:,} unique words, {t:,} total tokens, TTR={ttr:.3f}")


def compute_topic_level_stats(topics):
    """
    Compute statistics at topic level.

    Returns DataFrame with:
        - subject
        - topic_id
        - total_words: raw word count (before preprocessing)
        - unique_tokens: vocabulary size (after preprocessing)
    """
    records = []

    for (subject, topic_id), data in topics.items():
        full_text = data.get("full_text", "")

        # Raw word count (before preprocessing)
        total_words = len(full_text.split())

        # Unique tokens (after preprocessing)
        tokens = tokenize_normalize(full_text)
        unique_tokens = len(set(tokens))

        records.append({
            "subject": subject,
            "topic_id": topic_id,
            "total_words": total_words,
            "unique_tokens": unique_tokens,
        })

    df = pd.DataFrame(records)
    df = df.sort_values(["subject", "topic_id"]).reset_index(drop=True)
    return df


def plot_all_topics_bubble(topic_stats_df, outpath=None):
    """
    Bubble chart with 3 subplots (one per subject).
    Y: topic_id, X: total_words, Size: unique_tokens
    """
    subjects = sorted(topic_stats_df["subject"].unique())
    colors = {"DMA": "#4C72B0", "FCS": "#55A868", "FSD": "#C44E52"}

    # Find max topics per subject for figure height
    max_topics = max(len(topic_stats_df[topic_stats_df["subject"] == s]) for s in subjects)
    fig_height = max(6, max_topics * 0.5)

    fig, axes = plt.subplots(1, 3, figsize=(18, fig_height), sharex=True)

    # Global max for consistent bubble sizing
    max_unique = topic_stats_df["unique_tokens"].max()
    min_unique = topic_stats_df["unique_tokens"].min()

    for ax, subject in zip(axes, subjects):
        df = topic_stats_df[topic_stats_df["subject"] == subject].copy()
        df = df.sort_values("topic_id")

        total_words = df["total_words"].values
        unique_tokens = df["unique_tokens"].values
        topic_ids = df["topic_id"].values

        y_pos = np.arange(len(topic_ids))

        # Scale bubble size
        sizes = unique_tokens / max_unique * 800 + 100

        ax.scatter(total_words, y_pos, s=sizes,
                   c=colors.get(subject, "#999999"), alpha=0.6,
                   edgecolors="black", linewidths=1)

        # Add unique_tokens value on each bubble
        for tw, yp, ut in zip(total_words, y_pos, unique_tokens):
            ax.annotate(str(ut), (tw, yp), ha="center", va="center",
                       fontsize=7, fontweight="bold")

        ax.set_yticks(y_pos)
        ax.set_yticklabels(topic_ids)
        ax.set_xlabel("Total Words")
        ax.set_title(subject)
        ax.grid(axis="x", alpha=0.3)

    axes[0].set_ylabel("Topic")
    fig.suptitle("Topic Statistics by Subject", fontsize=12, y=1.02)

    # Add size legend
    legend_sizes = [min_unique, (min_unique + max_unique) // 2, max_unique]
    legend_bubbles = []
    for s in legend_sizes:
        size = s / max_unique * 800 + 100
        legend_bubbles.append(axes[2].scatter([], [], s=size, c="gray", alpha=0.6,
                                               edgecolors="black", linewidths=1))
    axes[2].legend(legend_bubbles, [f"{s} unique" for s in legend_sizes],
                   title="Bubble size", loc="lower right", fontsize=8)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    if outpath:
        Path(outpath).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()


# Main entry point

In [ ]:
# 1) Build topic-level corpus
topics = build_topic_corpus()

if not topics:
  print("[ERROR] No topics were built. Please check file names and paths.")

In [ ]:
import pickle
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/topics_{stamp}.pkl"

with open(path, "wb") as f:
    pickle.dump(topics, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved topics to:", path)
print("Topics:", len(topics))


In [ ]:
# 2) Topic-level statistics -> CSV
topics_df = summarize_topics(topics)

# 3) Vocabulary + n-gram analysis -> CSVs
subject_token_counter, subject_bigram_counter, subject_trigram_counter, top_tokens_df, top_bigrams_df, top_trigrams_df = analyze_vocabulary(topics, out_dir=OUTPUT_DIR)

# 4) Corpus statistics per subject
print("=== Corpus Statistics (after preprocessing) ===")
corpus_stats_df = summarize_corpus_by_subject(topics_df, subject_token_counter)
display(corpus_stats_df)

In [ ]:
# 5) Plots

for subject in sorted(topics_df["subject"].unique()):
    # 1) lengths distribution (topic-level)
    lengths = topics_df.loc[topics_df["subject"] == subject, "total_words"].tolist()
    plot_length_distribution(
        lengths,
        title=f"Distribution of topic lengths ({subject})",
        outpath=FIGS_DIR / f"{subject}_length_distribution.png",
    )

    tokens = subject_token_counter.get(subject, [])
    plot_wordcloud(
        tokens,
        title=f"Word cloud for {subject} teaching materials",
        outpath=FIGS_DIR / f"{subject}_wordcloud.png",
    )

    plot_top_words(
        tokens,
        title=f"Top 20 content words in {subject} corpus",
        outpath=FIGS_DIR / f"{subject}_top_words.png",
        top_k=20,
    )

    bg_ctr = subject_bigram_counter.get(subject, Counter())
    plot_top_ngrams_from_counter(
        bg_ctr,
        title=f"Top 20 bigrams in {subject} corpus",
        outpath=FIGS_DIR / f"{subject}_top_bigrams.png",
        top_k=20,
    )

    tg_ctr = subject_trigram_counter.get(subject, Counter())
    plot_top_ngrams_from_counter(
        tg_ctr,
        title=f"Top 20 trigrams in {subject} corpus",
        outpath=FIGS_DIR / f"{subject}_top_trigrams.png",
        top_k=20,
    )



In [ ]:
# Load pre-computed data (skip full pipeline)
import pickle

TOPICS_PKL = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/topics_20251230_094943.pkl"
TOPICS_CSV = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/analysis_topic_level/topics_summary.csv"

# Load topics dict
with open(TOPICS_PKL, "rb") as f:
    topics = pickle.load(f)
print(f"Loaded {len(topics)} topics from pickle")

# Load topics_df
topics_df = pd.read_csv(TOPICS_CSV)
print(f"Loaded topics_df with {len(topics_df)} rows")

# Recompute subject_token_counter (fast)
subject_token_counter, subject_bigram_counter, subject_trigram_counter, _, _, _ = analyze_vocabulary(topics, out_dir=None, verbose=False)
print("Recomputed vocabulary counters")

# Create corpus_stats_df for plots
corpus_stats_df = summarize_corpus_by_subject(topics_df, subject_token_counter)
display(corpus_stats_df)

In [ ]:
# 6) Corpus comparison plots

# Vocabulary size comparison (bar chart)
plot_vocabulary_size(
    subject_token_counter,
    outpath=FIGS_DIR / "vocabulary_size_comparison.png",
)

# 7) Topic-level bubble chart (all subjects)
topic_stats_df = compute_topic_level_stats(topics)
display(topic_stats_df)

plot_all_topics_bubble(
    topic_stats_df,
    outpath=FIGS_DIR / "all_topics_bubble.png",
)

# QA Generation

In [ ]:
!pip -q install transformers accelerate sentencepiece pandas tqdm



In [ ]:
import os, re, json, math, pickle
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/topics_20251230_094943.pkl"

with open(path, "rb") as f:
    topics = pickle.load(f)

print("Loaded topics:", len(topics))
print("Example key:", list(topics.keys())[0])

# Inspect topics structure
print(type(topics))
print("len:", len(topics))
keys = list(topics.keys())
print("first keys:", keys[:10])

first_key = keys[0]
first_val = topics[first_key]
print("first_key:", first_key)
print("type(value):", type(first_val))
print("value keys (first 50):", list(first_val.keys())[:50])


In [ ]:
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"
os.makedirs(OUT_DIR, exist_ok=True)

# Generation safety
MAX_NEW_TOKENS = 512
DO_SAMPLE = False  # deterministic generation

# How many QAs we ask per model call (bigger -> more JSON failures)
MAX_Q_PER_CALL = 4
# How many calls we allow per (chunk, mode) before giving up and moving on
MAX_CALLS_PER_CHUNK_MODE = 2
# If model returns fewer than requested in a call, we stop that mode for that chunk (move on)
STOP_MODE_IF_UNDERPRODUCE = True

# Quality filters (tune if needed)
MIN_Q_WORDS = 4
MIN_A_WORDS = 20
MAX_Q_CHARS = 220
MAX_A_CHARS = 1200

PLACEHOLDER_PATTERNS = [
    r"\bquestion here\b", r"\banswer here\b", r"\bfill in\b", r"\btbd\b",
]
PLACEHOLDER_RE = re.compile("|".join(PLACEHOLDER_PATTERNS), re.IGNORECASE)

# Universal modes (work for any subject)
QUESTION_MODES = [
    {"id": "core_concepts", "instruction": "Key concepts and main ideas; include why it matters."},
    {"id": "definitions_terms", "instruction": "Definitions and terminology; meaning, purpose, distinguishing features."},
    {"id": "apply_reason", "instruction": "Application and reasoning; small scenarios and 'what happens if' questions."},
    {"id": "compare_contrast", "instruction": "Compare/contrast two ideas from the text; trade-offs and when to use each."},
    {"id": "common_mistakes", "instruction": "Common mistakes/misconceptions and how to avoid them."},
]


In [ ]:
# Convert topics(dict) -> topic_items(list) + get text

from typing import Optional

TEXT_CANDIDATE_KEYS = [
    "full_text", "text", "content", "raw_text", "combined_text",
    "topic_text", "document_text", "body"
]

def extract_text_from_topic_dict(d: dict) -> Optional[str]:
    # 1) direct keys
    for k in TEXT_CANDIDATE_KEYS:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()

    # 2) nested dicts
    for outer_key in ["doc", "data", "document", "payload"]:
        outer = d.get(outer_key)
        if isinstance(outer, dict):
            for k in TEXT_CANDIDATE_KEYS:
                v = outer.get(k)
                if isinstance(v, str) and v.strip():
                    return v.strip()

    # 3) list of text parts
    for list_key in ["texts", "parts", "pages", "chunks_raw"]:
        v = d.get(list_key)
        if isinstance(v, list):
            joined = "\n\n".join([x for x in v if isinstance(x, str) and x.strip()])
            if joined.strip():
                return joined.strip()

    return None

def extract_files_from_topic_dict(d: dict):
    for k in ["source_files", "files", "file_list", "sources"]:
        v = d.get(k)
        if isinstance(v, list):
            return v
    return None

topic_items = []
missing_text = []

for (subject, topic_id), payload in topics.items():
    txt = extract_text_from_topic_dict(payload) if isinstance(payload, dict) else None
    files = extract_files_from_topic_dict(payload) if isinstance(payload, dict) else None

    topic_items.append({
        "subject": subject,
        "topic_id": topic_id,
        "full_text": txt or "",
        "source_files": files or []
    })

    if not txt:
        missing_text.append((subject, topic_id))

print("topic_items:", len(topic_items))
print("topics with missing text:", len(missing_text))
print("sample preview:", topic_items[0]["topic_id"], topic_items[0]["full_text"][:200])

# Topic-level stats
df_topics = pd.DataFrame(topic_items)
df_topics["words"] = df_topics["full_text"].apply(lambda s: len(s.split()) if isinstance(s, str) else 0)
df_topics["chars"] = df_topics["full_text"].apply(lambda s: len(s) if isinstance(s, str) else 0)

print("Topics per subject:")
print(df_topics.groupby("subject")["topic_id"].nunique())
print("\nWords per topic (by subject):")
display(df_topics.groupby("subject")["words"].describe())

df_topics.to_csv("topics_summary_from_topics.csv", index=False)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi


In [ ]:
# Load model + tokenizer
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

def guess_context_tokens(tok) -> int:
    mx = getattr(tok, "model_max_length", None)
    if mx is None or mx > 200_000:
        return 4096
    return int(mx)

CONTEXT_TOKENS = guess_context_tokens(tokenizer)
print("Context tokens:", CONTEXT_TOKENS)


In [ ]:
# Chunking by tokens (topic -> chunks)
CHUNK_TOKENS = 1200
OVERLAP_TOKENS = 120

def split_paragraphs(text: str):
    parts = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(parts) <= 1:
        parts = [p.strip() for p in text.splitlines() if p.strip()]
    return parts

def count_tokens(s: str) -> int:
    return len(tokenizer.encode(s, add_special_tokens=False))

def chunk_text(text: str, chunk_tokens: int, overlap_tokens: int):
    paras = split_paragraphs(text)
    chunks = []
    cur, cur_tokens = [], 0

    for p in paras:
        pt = count_tokens(p)

        # If a single paragraph is too large, split by sentences.
        if pt > chunk_tokens:
            sentences = re.split(r"(?<=[.!?])\s+", p)
            for sent in sentences:
                st = count_tokens(sent)
                if cur and cur_tokens + st > chunk_tokens:
                    chunk_str = "\n\n".join(cur).strip()
                    chunks.append({"text": chunk_str, "tokens": count_tokens(chunk_str)})

                    if overlap_tokens > 0:
                        enc = tokenizer.encode(chunk_str, add_special_tokens=False)
                        tail = enc[-overlap_tokens:]
                        cur = [tokenizer.decode(tail)]
                        cur_tokens = len(tail)
                    else:
                        cur, cur_tokens = [], 0

                cur.append(sent)
                cur_tokens += st
            continue

        if cur and cur_tokens + pt > chunk_tokens:
            chunk_str = "\n\n".join(cur).strip()
            chunks.append({"text": chunk_str, "tokens": count_tokens(chunk_str)})

            if overlap_tokens > 0:
                enc = tokenizer.encode(chunk_str, add_special_tokens=False)
                tail = enc[-overlap_tokens:]
                cur = [tokenizer.decode(tail)]
                cur_tokens = len(tail)
            else:
                cur, cur_tokens = [], 0

        cur.append(p)
        cur_tokens += pt

    if cur:
        chunk_str = "\n\n".join(cur).strip()
        chunks.append({"text": chunk_str, "tokens": count_tokens(chunk_str)})

    return chunks

topic_chunks = []
for t in topic_items:
    text = t["full_text"]
    if not isinstance(text, str) or not text.strip():
        continue

    chunks = chunk_text(text, CHUNK_TOKENS, OVERLAP_TOKENS)
    for i, ch in enumerate(chunks):
        topic_chunks.append({
            "subject": t["subject"],
            "topic_id": t["topic_id"],
            "chunk_id": i,
            "chunk_tokens": ch["tokens"],
            "chunk_text": ch["text"],
            "source_files": t["source_files"],
        })

df_chunks = pd.DataFrame(topic_chunks)
print("Total chunks:", len(df_chunks))
print(df_chunks["chunk_tokens"].describe())

# df_chunks.to_csv("chunk_index.csv", index=False)
df_chunks.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag/chunk_index.csv",
    index=False
)
df_chunks.head()


In [ ]:
BASE_RAG_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag"

for subject in ["fsd", "fcs", "dma"]:
    sub_df = df_chunks[df_chunks["subject"].str.lower() == subject]

    subject_dir = os.path.join(BASE_RAG_DIR, subject)
    os.makedirs(subject_dir, exist_ok=True)

    out_path = os.path.join(subject_dir, "chunk_index.csv")
    sub_df.to_csv(out_path, index=False)

    print(f"Saved {len(sub_df)} chunks to {out_path}")

In [ ]:
import os, pandas as pd

BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"
RAG_DIR = f"{BASE_PATH}/rag"
EVAL_QA_DIR = f"{BASE_PATH}/eval_qa"
os.makedirs(EVAL_QA_DIR, exist_ok=True)

SUBJECTS = ["fsd", "fcs", "dma"]
SEED = 42
EVAL_RATIO = 0.15
SPLIT_LEVEL = "chunk"

In [ ]:
def load_chunks_for_subject(subject: str) -> pd.DataFrame:
    path = f"{RAG_DIR}/{subject}/chunk_index.csv"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing chunk_index.csv: {path}")
    df = pd.read_csv(path)

    df["subject"] = df.get("subject", subject.upper())
    df["chunk_id"] = df["chunk_id"].astype(int)
    df["topic_id"] = df["topic_id"].astype(str)
    df["chunk_text"] = df["chunk_text"].astype(str)
    return df

In [ ]:
# Split chunks into train/eval
def add_split_column_single_subject(df: pd.DataFrame, seed=42, eval_ratio=0.15, split_level="chunk") -> pd.DataFrame:
    d = df.copy()

    if split_level == "topic":
        keys = d[["topic_id"]].drop_duplicates().sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_eval = max(1, int(len(keys) * eval_ratio))
        eval_topics = set(keys.head(n_eval)["topic_id"].tolist())
        d["split"] = d["topic_id"].apply(lambda t: "eval" if t in eval_topics else "train")
    else:
        keys = d[["topic_id", "chunk_id"]].drop_duplicates().sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_eval = max(1, int(len(keys) * eval_ratio))
        eval_keys = set(map(tuple, keys.head(n_eval).values.tolist()))
        d["split"] = d.apply(lambda r: "eval" if (r["topic_id"], int(r["chunk_id"])) in eval_keys else "train", axis=1)

    return d


In [ ]:
EVAL_QA_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/eval_qa"
os.makedirs(EVAL_QA_DIR, exist_ok=True)

In [ ]:
# QA targets + prompt builder
def questions_for_chunk(tokens: int) -> int:
    # universal scaling; tune later
    if tokens < 300:
        return 4
    if tokens < 700:
        return 8
    return 12

def build_prompt(subject: str, text: str, n_target: int, mode_instruction: str, tag: str) -> str:
    return f"""
You MUST output ONLY valid JSON.
No markdown. No explanations. No extra text.
Return a JSON array of objects.
Each object MUST contain exactly:
- "question"
- "answer"
Target: generate up to {n_target} question-answer pairs.
If you cannot produce exactly {n_target}, produce as many as you can (at least 1), up to {n_target}.
Do NOT include empty questions or empty answers.

Answer style:
- 2–6 sentences
- clear and correct
- do NOT invent facts not present in the STUDY TEXT

Question style requirement:
{mode_instruction}

Tag: {tag} (avoid repeating questions across different tags)

Example:
[
  {{"question":"What is X and why is it important?","answer":"X is ... It matters because ..."}}
]

STUDY TEXT:
{text}
""".strip()

In [ ]:
# Utilities + generate_text
def normalize_source_files(files):
    out = []
    for x in files or []:
        if isinstance(x, str):
            out.append(x)
        elif isinstance(x, dict):
            out.append(x.get("file_name") or x.get("source") or str(x))
        else:
            out.append(str(x))
    return out

def norm_question(q: str) -> str:
    q = q.lower().strip()
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"[^\w\s]", "", q)  # remove punctuation
    return q

def word_count(s: str) -> int:
    return len(re.findall(r"\b\w+\b", s))

def passes_quality(q: str, a: str) -> bool:
    if not q or not a:
        return False
    if PLACEHOLDER_RE.search(q) or PLACEHOLDER_RE.search(a):
        return False
    if word_count(q) < MIN_Q_WORDS:
        return False
    if word_count(a) < MIN_A_WORDS:
        return False
    if len(q) > MAX_Q_CHARS:
        return False
    if len(a) > MAX_A_CHARS:
        return False
    return True

def generate_text(prompt: str, model, tokenizer, max_new_tokens: int = MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,           # deterministic because DO_SAMPLE=False
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


In [ ]:
# JSON parse + clean
def try_parse_json_strict(text: str):
    decoder = json.JSONDecoder()

    for i, ch in enumerate(text):
        if ch not in "[{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
        except Exception:
            continue

        # normalize to list
        if isinstance(obj, dict):
            candidates = [obj]
        elif isinstance(obj, list):
            candidates = obj
        else:
            continue

        # accept only if it contains at least one QA-like object
        if any(isinstance(x, dict) and "question" in x and "answer" in x for x in candidates):
            return candidates

    return None

def clean_qa_objects(objs):
    cleaned = []
    for obj in objs or []:
        q_raw = obj.get("question", None)
        if q_raw is None:
            q_raw = obj.get("question ", None)
        a_raw = obj.get("answer", None)
        if q_raw is None or a_raw is None:
            continue
        q = str(q_raw).strip()
        a = str(a_raw).strip()
        if passes_quality(q, a):
            cleaned.append({"question": q, "answer": a})
    return cleaned

In [ ]:
import ast

def parse_source_files_cell(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.strip():
        try:
            v = ast.literal_eval(x)
            return v if isinstance(v, list) else [v]
        except Exception:
            return [x]
    return []

In [ ]:
# Main generation loop
def run_generation_for_subject(df_chunks: pd.DataFrame, subject: str, split: str, modes=QUESTION_MODES):
    df_s = df_chunks[(df_chunks["subject"] == subject) & (df_chunks["split"] == split)].copy()
    df_s["target_questions"] = df_s["chunk_tokens"].apply(questions_for_chunk)

    qa_rows = []
    errors = []

    # Global dedup within subject
    seen_q = set()

    for _, row in tqdm(df_s.iterrows(), total=len(df_s), desc=f"Generating QA for {subject}"):
        src_files = normalize_source_files(parse_source_files_cell(row["source_files"]))
        src_join = ";".join(src_files)

        for mode in modes:
            total_target = int(row["target_questions"])
            remaining = total_target
            calls_used = 0
            batch_index = 0

            while remaining > 0 and calls_used < MAX_CALLS_PER_CHUNK_MODE:
                calls_used += 1
                batch_index += 1

                n_req = min(MAX_Q_PER_CALL, remaining)
                tag = f"{subject}:{row['topic_id']}:c{int(row['chunk_id'])}:{mode['id']}:b{batch_index}"
                prompt = build_prompt(subject, row["chunk_text"], n_req, mode["instruction"], tag)

                raw = generate_text(prompt, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
                objs = try_parse_json_strict(raw)

                if objs is None:
                    repair_prompt = prompt + "\n\nYour previous output was invalid JSON. Output ONLY the JSON array."
                    raw2 = generate_text(repair_prompt, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
                    objs = try_parse_json_strict(raw2)
                    raw = raw2

                if objs is None:
                    errors.append({
                        "subject": subject,
                        "topic_id": row["topic_id"],
                        "chunk_id": int(row["chunk_id"]),
                        "chunk_tokens": int(row["chunk_tokens"]),
                        "mode": mode["id"],
                        "batch": batch_index,
                        "error": "Could not parse model output as JSON (after retry).",
                        "raw_output": raw[:2000],
                    })
                    break

                cleaned = clean_qa_objects(objs)

                # Dedup (exact-ish)
                final = []
                for qa in cleaned:
                    nq = norm_question(qa["question"])
                    if nq in seen_q:
                        continue
                    seen_q.add(nq)
                    final.append(qa)

                if len(final) == 0:
                    errors.append({
                        "subject": subject,
                        "topic_id": row["topic_id"],
                        "chunk_id": int(row["chunk_id"]),
                        "chunk_tokens": int(row["chunk_tokens"]),
                        "mode": mode["id"],
                        "batch": batch_index,
                        "error": "No valid QA objects after cleaning/dedup",
                        "raw_output": raw[:2000],
                    })
                    break

                for j, qa in enumerate(final):
                    qa_rows.append({
                        "subject": subject,
                        "topic_id": row["topic_id"],
                        "chunk_id": int(row["chunk_id"]),
                        "chunk_tokens": int(row["chunk_tokens"]),
                        "mode": mode["id"],
                        "batch": batch_index,
                        "qa_id": j,
                        "question": qa["question"],
                        "answer": qa["answer"],
                        "source_files": src_join,
                    })

                produced = len(final)
                remaining -= produced

                # If model under-produced, don't get stuck trying forever
                if STOP_MODE_IF_UNDERPRODUCE and produced < n_req:
                    break

    qa_df = pd.DataFrame(qa_rows)
    err_df = pd.DataFrame(errors)

    qa_path = f"{EVAL_QA_DIR}/{subject.lower()}_{split}.jsonl"
    err_path = f"{EVAL_QA_DIR}/qa_errors_{subject.lower()}_{split}.csv"
    stats_path = f"{EVAL_QA_DIR}/qa_stats_{subject.lower()}_{split}.csv"
    mode_stats_path = f"{EVAL_QA_DIR}/qa_stats_{subject.lower()}_{split}_by_mode.csv"


    qa_df.to_json(qa_path, orient="records", lines=True, force_ascii=False)
    err_df.to_csv(err_path, index=False)

    stats = {
        "subject": subject,
        "topics": df_s["topic_id"].nunique(),
        "chunks": len(df_s),
        "qas": len(qa_df),
        "errors": len(err_df),
        "chunk_tokens_min": int(df_s["chunk_tokens"].min()) if len(df_s) else 0,
        "chunk_tokens_median": float(df_s["chunk_tokens"].median()) if len(df_s) else 0,
        "chunk_tokens_mean": float(df_s["chunk_tokens"].mean()) if len(df_s) else 0,
        "chunk_tokens_max": int(df_s["chunk_tokens"].max()) if len(df_s) else 0,
    }

    stats_df = pd.DataFrame([stats])
    stats_df.to_csv(stats_path, index=False)

    if len(qa_df):
        by_mode = qa_df.groupby("mode").size().reset_index(name="qas")
        by_mode.to_csv(mode_stats_path, index=False)

    print(f"\n=== {subject} RESULTS ===")
    print(stats_df.to_string(index=False))
    if len(qa_df):
        print(pd.read_csv(mode_stats_path).to_string(index=False))
    print(f"Saved: {qa_path}\n {err_path}\n {stats_path}")

    return qa_df, err_df, stats_df

In [ ]:
def run_subject_pipeline(subject: str):
    df = load_chunks_for_subject(subject)
    df = add_split_column_single_subject(df, seed=SEED, eval_ratio=EVAL_RATIO, split_level=SPLIT_LEVEL)

    print(f"\n===== {subject.upper()} =====")
    print(df["split"].value_counts().to_dict())

    subject_label = subject.upper()

    qa_train, err_train, stats_train = run_generation_for_subject(df, subject_label, split="train")
    qa_eval,  err_eval,  stats_eval  = run_generation_for_subject(df, subject_label, split="eval")

    return {
        "df_chunks": df,
        "train": {"qa": qa_train, "err": err_train, "stats": stats_train},
        "eval":  {"qa": qa_eval,  "err": err_eval,  "stats": stats_eval},
    }

In [ ]:
# Run per subject
results = {}
# results["fsd"] = run_subject_pipeline("fsd")
# results["fcs"] = run_subject_pipeline("fcs")
results["dma"] = run_subject_pipeline("dma")

# QA Review

In [ ]:
import json, itertools

def peek_jsonl(path, n=3):
    with open(path, "r", encoding="utf-8") as f:
        for line in itertools.islice(f, n):
            obj = json.loads(line)
            print(obj["question"])
            print(obj["answer"][:200], "...\n")

peek_jsonl("/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/eval_qa/fsd_train.jsonl", n=2)
peek_jsonl("/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/eval_qa/fsd_eval.jsonl", n=2)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, ast
import pandas as pd

# Pandas display: show FULL text in cells
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)

# Base path
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"
EVAL_QA_DIR = os.path.join(BASE_DIR, "eval_qa")

# Helpers: read jsonl / json
def read_jsonl(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def read_json(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    return pd.DataFrame(obj)

# ID builders (robust)
def ensure_qa_id_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure 'qa_id' exists. If not present, create stable qa_id by row index.
    """
    if "qa_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["qa_id"] = df.index.astype(int)
    else:
        df["qa_id"] = df["qa_id"].astype(int)
    return df

def make_id_from_qa_row(r) -> str:
    """
    MUST match review ids:
    FSD_Topic_01-1-core_concepts-b1-0
    """
    topic_id = r.get("topic_id")
    chunk_id = int(r.get("chunk_id"))
    mode = r.get("mode", "unknown")
    batch = int(r.get("batch"))
    qa_id = int(r.get("qa_id"))
    return f"{topic_id}-{chunk_id}-{mode}-b{batch}-{qa_id}"

def make_alt_ids(subject: str, split: str, r) -> dict:
    base = make_id_from_qa_row(r)
    alt2 = f"{subject.upper()}_{base}"
    alt3 = f"{subject.upper()}:{base}"
    return {"id_base": base, "id_alt2": alt2, "id_alt3": alt3}

def parse_source_files_cell(x):
    """
    Your jsonl might store source_files as:
    - list
    - stringified list (python-like or json-like)
    - plain string
    We'll normalize to list for nicer viewing/exports.
    """
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.strip():
        t = x.strip()
        # Try JSON first
        try:
            v = json.loads(t)
            return v if isinstance(v, list) else [v]
        except Exception:
            pass
        # Try python literal
        try:
            v = ast.literal_eval(t)
            return v if isinstance(v, list) else [v]
        except Exception:
            return [t]
    return []

def normalize_source_files(sf_list) -> list:
    out = []
    for item in sf_list or []:
        if isinstance(item, dict):
            # Prefer file_name if present
            out.append(str(item.get("file_name") or item))
        else:
            out.append(str(item))
    return out

def print_qa_cards(df: pd.DataFrame, n: int = 10):
    """Print full Q/A for first n rows (no truncation)."""
    for _, r in df.head(n).iterrows():
        print("ID:", r.get("id"))
        print("Label:", r.get("label"))
        print("Reasons:", r.get("reasons"))
        print("Notes:", r.get("notes"))
        print("\nQ:", r.get("question"))
        print("\nA:", r.get("answer"))
        if "source_files" in df.columns:
            print("\nSources:", r.get("source_files"))
        print("\n" + "-"*120 + "\n")


def export_ok_and_flag_for_subject(subject: str, split: str, base_dir: str = BASE_DIR):
    qa_dir = os.path.join(base_dir, "eval_qa")

    qa_path     = os.path.join(qa_dir, f"{subject.lower()}_{split}.jsonl")
    review_path = os.path.join(qa_dir, f"{subject.lower()}_{split}_review.json")

    if not os.path.exists(qa_path):
        raise FileNotFoundError(f"QA file not found: {qa_path}")
    if not os.path.exists(review_path):
        raise FileNotFoundError(f"Review file not found: {review_path}")

    # Load
    qa = read_jsonl(qa_path)
    review = read_json(review_path)

    # Ensure required QA columns exist (batch optional)
    required = ["topic_id", "chunk_id", "mode", "question", "answer"]
    missing = [c for c in required if c not in qa.columns]
    if missing:
        raise ValueError(f"QA file missing columns: {missing}")

    # Ensure qa_id exists
    qa = ensure_qa_id_column(qa)

    # Normalize source_files if present
    if "source_files" in qa.columns:
        qa["source_files"] = qa["source_files"].apply(parse_source_files_cell).apply(normalize_source_files)

    # Build join ids in QA
    ids = qa.apply(lambda r: make_alt_ids(subject, split, r), axis=1, result_type="expand")
    qa = pd.concat([qa, ids], axis=1)

    # Review must have id
    if "id" not in review.columns:
        raise ValueError("Review JSON must contain an 'id' column to join.")

    review_cols = [c for c in ["id", "label", "reasons", "notes"] if c in review.columns]
    review_small = review[review_cols].copy()

    # Merge
    merged = qa.merge(review_small, left_on="id_base", right_on="id", how="left")
    missing_rate = merged["label"].isna().mean()


    if missing_rate > 0.5:
        merged_alt2 = qa.merge(review_small, left_on="id_alt2", right_on="id", how="left")
        if merged_alt2["label"].isna().mean() < missing_rate:
            merged = merged_alt2
            missing_rate = merged["label"].isna().mean()

    if missing_rate > 0.5:
        merged_alt3 = qa.merge(review_small, left_on="id_alt3", right_on="id", how="left")
        if merged_alt3["label"].isna().mean() < missing_rate:
            merged = merged_alt3
            missing_rate = merged["label"].isna().mean()

    # Clean id
    merged["id"] = merged["id"].fillna(merged["id_base"])

    # Summary
    total = len(merged)
    n_ok = (merged["label"] == "OK").sum()
    n_flag = (merged["label"] == "FLAG").sum()
    n_unlabeled = merged["label"].isna().sum()

    print(f"\n=== {subject.upper()} ({split}) REVIEW SUMMARY ===")
    print(f"Total QA: {total}")
    print(f"OK:      {n_ok}")
    print(f"FLAG:    {n_flag}  ({(n_flag/total):.1%} of total)")
    print(f"No label (unmatched ids): {n_unlabeled}  ({(n_unlabeled/total):.1%})")
    if n_unlabeled > 0:
        print("Note: Some review IDs did not match QA IDs. Check ID format if this is unexpected.")

    # Columns to export
    cols = [c for c in [
        "id", "topic_id", "chunk_id", "mode", "batch", "qa_id",
        "question", "answer", "label", "reasons", "notes", "source_files"
    ] if c in merged.columns]

    # Split into OK / FLAG
    df_ok = merged[merged["label"] == "OK"].copy()
    df_flag = merged[merged["label"] == "FLAG"].copy()

    # Save both
    out_ok_csv   = os.path.join(qa_dir, f"{subject.lower()}_{split}_OK.csv")
    out_ok_txt   = os.path.join(qa_dir, f"{subject.lower()}_{split}_OK.txt")
    out_flag_csv = os.path.join(qa_dir, f"{subject.lower()}_{split}_FLAGGED.csv")
    out_flag_txt = os.path.join(qa_dir, f"{subject.lower()}_{split}_FLAGGED.txt")
    out_ok_jsonl = os.path.join(qa_dir, f"{subject.lower()}_{split}_OK.jsonl")
    out_flag_jsonl = os.path.join(qa_dir, f"{subject.lower()}_{split}_FLAGGED.jsonl")

    df_ok[cols].to_csv(out_ok_csv, index=False)
    df_flag[cols].to_csv(out_flag_csv, index=False)

    def df_to_jsonl(df: pd.DataFrame, path: str):
        with open(path, "w", encoding="utf-8") as f:
            for _, r in df.iterrows():
                rec = {
                    "subject": r.get("subject") or subject.upper(),
                    "topic_id": r.get("topic_id"),
                    "chunk_id": int(r.get("chunk_id")) if pd.notna(r.get("chunk_id")) else None,
                    "mode": r.get("mode"),
                    "batch": int(r.get("batch")) if "batch" in df.columns and pd.notna(r.get("batch")) else None,
                    "qa_id": int(r.get("qa_id")) if pd.notna(r.get("qa_id")) else None,
                    "question": r.get("question"),
                    "answer": r.get("answer"),
                    "source_files": r.get("source_files", []),
                }
                rec = {k: v for k, v in rec.items() if v is not None}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    df_to_jsonl(df_ok, out_ok_jsonl)
    df_to_jsonl(df_flag, out_flag_jsonl)

    def write_cards(df, path):
        with open(path, "w", encoding="utf-8") as f:
            for _, r in df[cols].iterrows():
                f.write(f"ID: {r.get('id')}\n")
                f.write(f"Label: {r.get('label')}\n")
                f.write(f"Reasons: {r.get('reasons')}\n")
                f.write(f"Notes: {r.get('notes')}\n")
                f.write(f"Q: {r.get('question')}\n")
                f.write(f"A: {r.get('answer')}\n")
                if "source_files" in df.columns:
                    f.write(f"Sources: {r.get('source_files')}\n")
                f.write("-"*140 + "\n\n")

    write_cards(df_ok, out_ok_txt)
    write_cards(df_flag, out_flag_txt)

    print("\nSaved files:")
    print("OK CSV   :", out_ok_csv)
    print("OK TXT   :", out_ok_txt)
    print("FLAG CSV :", out_flag_csv)
    print("FLAG TXT :", out_flag_txt)
    print("OK JSONL:", out_ok_jsonl)
    print("FLAG JSONL:", out_flag_jsonl)

    print("\nPreview OK (first 10):")
    display(df_ok[cols].head(10))
    print("\nPreview FLAG (first 10):")
    display(df_flag[cols].head(10))

    print("\nSample OK cards:")
    print_qa_cards(df_ok[cols], n=3)

    print("\nSample FLAG cards:")
    print_qa_cards(df_flag[cols], n=5)

    # Optional: reason breakdown for FLAG
    reason_counts = None
    if "reasons" in df_flag.columns:
        try:
            reasons_flat = df_flag.explode("reasons")["reasons"].dropna()
            reason_counts = reasons_flat.value_counts().reset_index()
            reason_counts.columns = ["reason", "count"]
            print("\nTop FLAG reasons:")
            display(reason_counts.head(50))
        except Exception:
            pass

    return df_ok[cols], df_flag[cols], reason_counts


In [ ]:
# ok_fsd_train, flag_fsd_train, reasons = export_ok_and_flag_for_subject("FSD", "train", BASE_DIR)
# ok_fcs_train, flag_fcs_train, reasons = export_ok_and_flag_for_subject("FCS", "train", BASE_DIR)
# ok_dma_train, flag_dma_train, reasons = export_ok_and_flag_for_subject("DMA", "train", BASE_DIR)
ok_fsd_eval, flag_fsd_eval, reasons = export_ok_and_flag_for_subject("FSD", "eval", BASE_DIR)
# ok_fcs_eval, flag_fcs_eval, reasons = export_ok_and_flag_for_subject("FCS", "eval", BASE_DIR)
# ok_dma_eval, flag_dma_eval, reasons = export_ok_and_flag_for_subject("DMA", "eval", BASE_DIR)

# Unsloth LORA

In [ ]:
# Install

%%capture
!pip -q uninstall -y unsloth unsloth_zoo unsloth-zoo trl transformers peft datasets pyarrow tyro msgspec torchao || true

# Colab-friendly core deps
!pip -q install "numpy==2.0.2"

# Unsloth stack
!pip -q install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip -q install sentencepiece protobuf "datasets==4.3.0" "pyarrow==21.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip -q install --no-deps unsloth
!pip -q install "transformers==4.56.2" --no-deps
!pip -q install "trl==0.22.2" --no-deps

# missing deps sometimes needed by unsloth-zoo
!pip -q install tyro msgspec "torchao>=0.13.0"


Now do: Runtime -> Restart runtime.

# RUN

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["DATASETS_DISABLE_MULTIPROCESSING"] = "1"


In [ ]:
# Imports

import unsloth
from unsloth import FastLanguageModel
import torch
import os, re, json

from unsloth.chat_templates import get_chat_template
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from peft import PeftModel
import numpy as np
import pandas as pd


In [ ]:
# Config
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"
EVAL_QA_DIR = os.path.join(BASE_DIR, "eval_qa")
ADAPTERS_DIR = os.path.join(BASE_DIR, "adapters")

def safe_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_\-]", "", s)
    return s

print("BASE_DIR:", BASE_DIR)
print("EVAL_QA_DIR exists?", os.path.exists(EVAL_QA_DIR))
print("ADAPTERS_DIR exists?", os.path.exists(ADAPTERS_DIR))

SUBJECT = "DMA"        # FSD | FCS | DMA
MODE = "TRAIN"         # TRAIN | EVAL  (TRAIN = LoRA fine-tuning)

SUBJECT_SAFE = safe_name(SUBJECT)
SPLIT = "train" if MODE == "TRAIN" else "eval"

QA_PATH = os.path.join(EVAL_QA_DIR, f"{SUBJECT_SAFE}_{SPLIT}_OK.jsonl")
assert os.path.exists(QA_PATH), f"QA file not found: {QA_PATH}"

BASE_MODEL_NAME = "microsoft/Phi-4-mini-instruct"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

# Where to save LoRA adapter for this subject
ADAPTER_OUT = os.path.join(ADAPTERS_DIR, f"lora_phi4mini_{SUBJECT}")
os.makedirs(ADAPTER_OUT, exist_ok=True)

print("SUBJECT:", SUBJECT, "| MODE:", MODE)
print("QA_PATH:", QA_PATH)
print("ADAPTER_OUT:", ADAPTER_OUT)


In [ ]:
# Load Phi-4-mini-instruct

max_seq_length = 2048
load_in_4bit = True

model_name = "microsoft/Phi-4-mini-instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
tokenizer = get_chat_template(tokenizer, chat_template="phi-4")


In [ ]:
# Load dataset + convert to chat format

def read_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = read_jsonl(QA_PATH)
print("Loaded rows:", len(rows))
print("Keys:", list(rows[0].keys()))

# Expect at least question/answer
assert "question" in rows[0] and "answer" in rows[0], "jsonl must contain question and answer fields"

SYSTEM = (
    "You are a helpful tutor. Answer clearly and step-by-step when appropriate. "
    "Output ONLY the answer to the question. "
    "Do NOT include repository names, metadata tags, or unrelated code. "
    "If code is required, output only minimal code relevant to the question."
)

# Convert into a "text" field using the model's chat template
# This is the standard format for SFT / LoRA.
def to_chat_text(example):
    messages = [
        {"role":"system", "content": SYSTEM},
        {"role":"user", "content": example["question"]},
        {"role":"assistant", "content": example["answer"]},
    ]
    # We'll set tokenizer later, but we can keep the messages now.
    example["messages"] = messages
    return example

rows = [to_chat_text(r) for r in rows]

ds = Dataset.from_list(rows)
print(ds)


In [ ]:
# Load base model + tokenizer (clean base)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
)

# Ensure the correct chat template for Phi
tokenizer = get_chat_template(tokenizer, chat_template="phi-4")

print("Loaded base model:", BASE_MODEL_NAME)


In [ ]:
# Attach LoRA
if isinstance(model, PeftModel):
    raise RuntimeError(
        "Model already has LoRA adapters attached. "
        "Restart runtime OR reload base model before calling get_peft_model."
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
)

print("LoRA attached.")


In [ ]:
# Build training text with chat template
def render_chat(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def add_text_field(example):
    example["text"] = render_chat(example["messages"])
    return example

ds = ds.map(add_text_field)
print("Sample text:\n", ds[0]["text"][:500])


In [ ]:
# Split (only for TRAIN mode)
# - This is an internal tiny validation split within TRAIN data.
# - Your real evaluation is SUBJECT_eval.jsonl (kept separate).

if MODE == "TRAIN":
    ds_split = ds.train_test_split(test_size=0.02, seed=42)
    train_ds = ds_split["train"]
    val_ds = ds_split["test"]
    print("Train:", len(train_ds), "| Val:", len(val_ds))
else:
    train_ds = None
    val_ds = None
    print("MODE=EVAL: not splitting. Use this dataset for metrics only.")


In [ ]:
# Train (SFT) - only if MODE == TRAIN
if MODE == "TRAIN":
    from trl import SFTTrainer
    from transformers import TrainingArguments

    training_args = TrainingArguments(
        output_dir=os.path.join(BASE_DIR, "runs", f"{SUBJECT_SAFE}_lora_phi4mini"),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        weight_decay=0.01,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        fp16=True if torch.cuda.is_available() else False,
        bf16=False,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        args=training_args,
    )

    trainer.train()

    # Save adapter
    model.save_pretrained(ADAPTER_OUT)
    tokenizer.save_pretrained(ADAPTER_OUT)

    print("Saved LoRA adapter to:", ADAPTER_OUT)


In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

STOP_MARKERS = [
    "<|im_end|>", "<|im_start|>",
    "<reponame>", "<gh_stars>",
    "#include", "```", "using namespace"
]

def cut_garbage(text: str) -> str:
    text = (text or "").strip()
    for m in STOP_MARKERS:
        idx = text.find(m)
        if idx != -1:
            text = text[:idx].strip()
            break
    return text

def generate_answer(question: str):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    attn = torch.ones_like(input_ids)
    eos = tokenizer.eos_token_id
    pad = eos if tokenizer.pad_token_id is None else tokenizer.pad_token_id


    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=160,
            temperature=0.3,
            do_sample=True,
            min_p=0.05,
            repetition_penalty=1.1,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return cut_garbage(text)


print(generate_answer("Explain the difference between declaration and initialization in C with an example."))


# RAG

In [ ]:
!pip -q install \
  chromadb \
  sentence-transformers \
  pandas \
  numpy

In [ ]:
# RAG (ChromaDB) single-subject version (1 container = 1 subject)
# CSV chunks -> embeddings -> Chroma persistent index -> retrieve -> build context

import os, shutil
import re
import json
from typing import List, Dict, Any, Tuple, Optional

import pandas as pd
import numpy as np

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer


# CONFIG
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

SUBJECT = "dma"  # "fsd" | "fcs" | "dma"  (single-subject container)

CHROMA_DIR = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag/{SUBJECT}/chroma_store"

# Local: Chroma (SQLite)
# CHROMA_DIR = f"/content/chroma_store_{SUBJECT}"

CHUNK_CSV_PATH = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag/{SUBJECT}/chunk_index.csv"

TOP_K = 6
MAX_CONTEXT_TOKENS = 1800


# UTILS
def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    return str(x)


def normalize_text(s: str) -> str:
    s = safe_str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s


def parse_list_field(x) -> List[str]:
    """
    Parses source_files stored in CSV/metadata as:
    - list
    - JSON string: '["a.pdf","b.pdf"]'
    - python-ish list string: "['a.pdf','b.pdf']"
    - plain string
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [safe_str(i) for i in x]
    if isinstance(x, str):
        t = x.strip()
        if not t:
            return []
        # try JSON list
        try:
            v = json.loads(t)
            if isinstance(v, list):
                return [safe_str(i) for i in v]
        except Exception:
            pass
        # python-ish list fallback
        if t.startswith("[") and t.endswith("]"):
            inner = t[1:-1].strip()
            if not inner:
                return []
            parts = [p.strip().strip("'\"") for p in inner.split(",")]
            return [p for p in parts if p]
        return [t]
    return [safe_str(x)]


def parse_source_files(x) -> List[str]:
    """
    Wrapper used by builder. Ensures we always return a list[str].
    """
    return parse_list_field(x)


def approximate_count_tokens(text: str) -> int:
    """
    Rough budget: ~1 token per 4 chars (English).
    Good enough for cutting context size.
    """
    text = safe_str(text)
    if not text:
        return 0
    return max(1, int(len(text) / 4))


def truncate_to_token_budget(
    chunks: List[Dict[str, Any]],
    max_tokens: int,
) -> Tuple[str, List[Dict[str, Any]]]:
    """
    chunks: list of {"text","meta","tokens"} sorted by relevance.
    Returns combined context + chunks used.
    """
    used = []
    total = 0
    parts = []

    for ch in chunks:
        t = ch.get("tokens")
        if t is None:
            t = approximate_count_tokens(ch.get("text", ""))

        if total + t > max_tokens and used:
            break

        txt = safe_str(ch.get("text", "")).strip()
        if txt:
            parts.append(txt)
            used.append(ch)
            total += t

        if total >= max_tokens:
            break

    return "\n\n---\n\n".join(parts), used


# CHROMA: init + collection
def get_chroma_client(persist_dir: str) -> chromadb.Client:
    os.makedirs(persist_dir, exist_ok=True)
    return chromadb.PersistentClient(
        path=persist_dir,
        settings=Settings(anonymized_telemetry=False),
    )


def get_collection(client: chromadb.Client):
    """
    Single-subject container => one collection name is enough.
    """
    return client.get_or_create_collection(
        name="rag",
        metadata={"hnsw:space": "cosine"},
    )


# BUILD INDEX
def load_chunks_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    needed = {"topic_id", "chunk_id", "chunk_text"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"chunk CSV missing columns: {missing}")

    if "chunk_tokens" not in df.columns:
        df["chunk_tokens"] = df["chunk_text"].astype(str).apply(approximate_count_tokens)
    if "source_files" not in df.columns:
        df["source_files"] = ""

    return df


def build_chroma_index_from_df(
    df_chunks: pd.DataFrame,
    embedder: SentenceTransformer,
    persist_dir: str,
    batch_size: int = 128,
) -> None:
    client = get_chroma_client(persist_dir)
    coll = get_collection(client)

    ids, docs, metas = [], [], []

    for _, row in df_chunks.iterrows():
        text = safe_str(row["chunk_text"]).strip()
        if not text:
            continue

        topic_id = safe_str(row["topic_id"])
        chunk_id = int(row["chunk_id"])
        chunk_tokens = int(row.get("chunk_tokens", approximate_count_tokens(text)))

        # include SUBJECT in ID for safety during development
        sid = f"{SUBJECT}::{topic_id}::{chunk_id}"
        ids.append(sid)
        docs.append(text)

        sf = parse_source_files(row.get("source_files"))
        metas.append({
            "topic_id": topic_id,
            "chunk_id": chunk_id,
            "chunk_tokens": chunk_tokens,
            "source_files": json.dumps(sf, ensure_ascii=False),
        })

    print(f"[build] indexing {len(ids)} chunks into {persist_dir} ...")

    for i in range(0, len(ids), batch_size):
        batch_docs = docs[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        batch_metas = metas[i:i+batch_size]

        embs = embedder.encode(
            batch_docs,
            batch_size=min(64, len(batch_docs)),
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32).tolist()

        # upsert = safe to re-run
        coll.upsert(
            ids=batch_ids,
            documents=batch_docs,
            metadatas=batch_metas,
            embeddings=embs,
        )

    print("[build] done.")


# RETRIEVAL
def retrieve_chunks(
    query: str,
    embedder: SentenceTransformer,
    persist_dir: str,
    top_k: int = TOP_K,
) -> List[Dict[str, Any]]:
    client = get_chroma_client(persist_dir)
    coll = get_collection(client)

    q = normalize_text(query)
    q_emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32).tolist()[0]

    res = coll.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    out: List[Dict[str, Any]] = []
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        out.append({
            "text": safe_str(doc),
            "meta": meta,
            "distance": float(dist) if dist is not None else None,
            "tokens": int(meta.get("chunk_tokens", approximate_count_tokens(doc))),
        })
    return out


def build_rag_context(
    query: str,
    embedder: SentenceTransformer,
    persist_dir: str,
    top_k: int = TOP_K,
    max_context_tokens: int = MAX_CONTEXT_TOKENS,
) -> Dict[str, Any]:
    hits = retrieve_chunks(
        query=query,
        embedder=embedder,
        persist_dir=persist_dir,
        top_k=top_k,
    )

    context_text, used = truncate_to_token_budget(hits, max_context_tokens)

    citations = []
    for i, ch in enumerate(used, start=1):
        meta = ch.get("meta", {}) or {}
        src_raw = meta.get("source_files", "[]")
        try:
            src_list = json.loads(src_raw) if isinstance(src_raw, str) else parse_list_field(src_raw)
        except Exception:
            src_list = parse_list_field(src_raw)

        citations.append({
            "idx": i,
            "topic_id": meta.get("topic_id"),
            "chunk_id": meta.get("chunk_id"),
            "source_files": src_list,
            "distance": ch.get("distance"),
        })

    return {
        "query": query,
        "subject": SUBJECT,
        "context": context_text,
        "chunks_used": used,
        "citations": citations,
    }


# USAGE
if __name__ == "__main__":
    # Load chunks for this SUBJECT
    df_chunks = load_chunks_csv(CHUNK_CSV_PATH)

    # Init embedder
    embedder = SentenceTransformer(EMBED_MODEL_NAME)

    # Build index
    if os.path.exists(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR)
    os.makedirs(CHROMA_DIR, exist_ok=True)

    build_chroma_index_from_df(df_chunks, embedder, persist_dir=CHROMA_DIR, batch_size=128)

    # Retrieve + build context
    question = "What is the difference between declaring and initializing a variable in C?"
    rag = build_rag_context(question, embedder, persist_dir=CHROMA_DIR, top_k=TOP_K, max_context_tokens=MAX_CONTEXT_TOKENS)

    print("\n=== RAG CONTEXT ===\n")
    print(rag["context"][:2000])

    print("\n=== CITATIONS ===\n")
    for c in rag["citations"]:
        print(c)


# SLM Check

In [ ]:
import importlib.metadata, torch
print("bitsandbytes =", importlib.metadata.version("bitsandbytes"))
print("torch =", torch.__version__, "cuda =", torch.version.cuda, "is_available =", torch.cuda.is_available())


In [ ]:
!pip -q install -U unsloth chromadb sentence-transformers peft accelerate bitsandbytes

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"


In [ ]:
import unsloth
import torch
import re
import json
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel

# =========================
# 3) Global config
# =========================
SUBJECT = "fsd"  # "fsd" | "fcs" | "dma"  (use lowercase)

BASE_MODEL_NAME = "microsoft/Phi-4-mini-instruct"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

ADAPTER_DIR = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/adapters/lora_phi4mini_{SUBJECT.upper()}"

CHUNK_CSV_PATH = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag/{SUBJECT}/chunk_index.csv"
CHROMA_DIR = f"/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor/rag/{SUBJECT}/chroma_store"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 6  # retrieval candidates (we will truncate by real token budget)
GEN_MAX_NEW_TOKENS = 420  # generation budget

SYSTEM = """You are a helpful C programming tutor.
Answer clearly and step-by-step when appropriate.
Return plain text only. Do NOT output any tool markers or markup tokens (e.g., <|tool_call|>, <|im_sep|>, <reponame>, LaTeX).
Important C fact: local variables are NOT implicitly initialized to 0; they have indeterminate values unless initialized.
"""

# =========================
# 4) Load BASE + LoRA
# =========================
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
)
tokenizer = get_chat_template(tokenizer, chat_template="phi-4")
FastLanguageModel.for_inference(base_model)

lora_base, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
)
lora_model = PeftModel.from_pretrained(lora_base, ADAPTER_DIR)
FastLanguageModel.for_inference(lora_model)

# =========================
# 5) Output cleaning + robust generation helper
# =========================
def clean_output(s: str) -> str:
    if not s:
        return ""
    s = s.replace("<br>", "\n").replace("<br/>", "\n").replace("<br />", "\n")

    # remove/stop on common junk markers
    # (keep conservative so we don't accidentally delete valid text)
    junk_markers = [
        "<|tool_call|>",
        "<|im_sep|>",
        "<|im_end|>",
        "<reponame>", "<gh_stars>",
        "\\documentclass", "\\usepackage",
        "<im_start>", "<im_end>",
    ]
    # First remove tool_call if it appears at the start
    s = s.replace("<|tool_call|>", "").strip()

    # Hard cut if any marker appears later
    for m in junk_markers:
        if m in s:
            s = s.split(m)[0]

    s = re.sub(r"\n{3,}", "\n\n", s).strip()
    return s

def _valid_stop_id(tok_id: int) -> bool:
    if tok_id is None or not isinstance(tok_id, int) or tok_id < 0:
        return False
    # avoid stopping on unk
    unk = getattr(tokenizer, "unk_token_id", None)
    if unk is not None and tok_id == unk:
        return False
    return True

def _encode_phrase(phrase: str):
    ids = tokenizer.encode(phrase, add_special_tokens=False)
    if not ids:
        return None
    unk = getattr(tokenizer, "unk_token_id", None)
    if unk is not None and all(i == unk for i in ids):
        return None
    return ids

def generate_answer(model, question: str, context: str = "") -> str:
    if context.strip():
        user_content = (
            "Use the following course material excerpts to answer.\n"
            "Answer for the C language (not C++).\n\n"
            f"COURSE MATERIAL:\n{context}\n\n"
            f"QUESTION:\n{question}"
        )
    else:
        user_content = "Answer for the C language (not C++).\n\n" + question

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    attention_mask = torch.ones_like(input_ids)

    # Stop tokens: use only VALID ids; do NOT mutate tokenizer vocab
    eos_ids = []
    if tokenizer.eos_token_id is not None:
        eos_ids.append(int(tokenizer.eos_token_id))

    for tok in ["<|im_end|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if _valid_stop_id(tid):
            eos_ids.append(int(tid))

    # de-dup
    eos_ids = list(dict.fromkeys(eos_ids))

    pad_id = tokenizer.eos_token_id if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    # Ban junk sequences (only if encodes meaningfully)
    bad_phrases = [
        "<|tool_call|>",
        "<|im_sep|>",
        "<|im_end|>",
        "<reponame>", "<gh_stars>",
        "\\documentclass", "\\usepackage",
        "<im_start>", "<im_end>",
    ]
    bad_words_ids = []
    for p in bad_phrases:
        enc = _encode_phrase(p)
        if enc is not None:
            bad_words_ids.append(enc)

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            temperature=0.4,
            min_p=0.05,
            repetition_penalty=1.05,
            use_cache=True,
            eos_token_id=eos_ids if len(eos_ids) > 1 else eos_ids[0],
            pad_token_id=pad_id,
            bad_words_ids=bad_words_ids,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Extra safety cut if some markers slip through decode
    for marker in ["<|tool_call|>", "<|im_sep|>", "<|im_end|>", "<reponame>", "\\documentclass"]:
        if marker in text:
            text = text.split(marker)[0]

    return clean_output(text.strip())

# =========================
# 6) RAG utilities (ChromaDB)
# =========================
def safe_str(x) -> str:
    if x is None:
        return ""
    return x if isinstance(x, str) else str(x)

def normalize_text(s: str) -> str:
    s = safe_str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def parse_list_field(x) -> List[str]:
    """
    Reads source_files stored as:
    - list
    - JSON string
    - python-like list string
    - plain string
    Returns a list[str].
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [safe_str(i) for i in x]
    if isinstance(x, str):
        t = x.strip()
        if not t:
            return []
        try:
            v = json.loads(t)
            if isinstance(v, list):
                return [safe_str(i) for i in v]
        except Exception:
            pass
        if t.startswith("[") and t.endswith("]"):
            inner = t[1:-1].strip()
            if not inner:
                return []
            parts = [p.strip().strip("'\"") for p in inner.split(",")]
            return [p for p in parts if p]
        return [t]
    return [safe_str(x)]

def stringify_metadata_value(v) -> str:
    """
    Chroma metadata values must be str/int/float/bool/None (NOT list/dict).
    Store lists/dicts as JSON strings.
    """
    if v is None:
        return ""
    if isinstance(v, (str, int, float, bool)):
        return v
    return json.dumps(v, ensure_ascii=False)

def get_chroma_client(persist_dir: str) -> chromadb.Client:
    os.makedirs(persist_dir, exist_ok=True)
    return chromadb.PersistentClient(
        path=persist_dir,
        settings=Settings(anonymized_telemetry=False),
    )

def get_collection(client: chromadb.Client) -> Any:
    return client.get_or_create_collection(
        name="rag",
        metadata={"hnsw:space": "cosine"},
    )

def load_chunks_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    needed = {"topic_id", "chunk_id", "chunk_text"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"chunk CSV missing columns: {missing}")

    if "chunk_tokens" not in df.columns:
        # keep it, but we won't rely on it for budgeting
        df["chunk_tokens"] = df["chunk_text"].astype(str).apply(lambda t: max(1, int(len(str(t))/4)))

    if "source_files" not in df.columns:
        df["source_files"] = ""

    return df

def build_chroma_index_from_df(
    df_chunks: pd.DataFrame,
    embedder: SentenceTransformer,
    persist_dir: str,
    batch_size: int = 128,
) -> None:
    client = get_chroma_client(persist_dir)
    coll = get_collection(client)

    ids, docs, metas = [], [], []

    for _, row in df_chunks.iterrows():
        text = safe_str(row["chunk_text"]).strip()
        if not text:
            continue

        topic_id = safe_str(row["topic_id"])
        chunk_id = int(row["chunk_id"])
        sid = f"{topic_id}::{chunk_id}"

        ids.append(sid)
        docs.append(text)

        sf = parse_list_field(row.get("source_files"))
        metas.append({
            "topic_id": topic_id,
            "chunk_id": chunk_id,
            "chunk_tokens": int(row.get("chunk_tokens", max(1, int(len(text)/4)))),
            "source_files": stringify_metadata_value(sf),
        })

    print(f"[build] indexing {len(ids)} chunks into {persist_dir} ...")

    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        batch_docs = docs[i:i+batch_size]
        batch_metas = metas[i:i+batch_size]

        embs = embedder.encode(
            batch_docs,
            batch_size=min(64, len(batch_docs)),
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32).tolist()

        coll.upsert(
            ids=batch_ids,
            documents=batch_docs,
            metadatas=batch_metas,
            embeddings=embs,
        )

    print("[build] done.")

def retrieve_chunks(
    query: str,
    embedder: SentenceTransformer,
    persist_dir: str,
    top_k: int = TOP_K,
) -> List[Dict[str, Any]]:
    client = get_chroma_client(persist_dir)
    coll = get_collection(client)

    q = normalize_text(query)
    q_emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32).tolist()[0]

    res = coll.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    hits = []
    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        hits.append({
            "text": safe_str(doc),
            "meta": meta,
            "distance": float(dist) if dist is not None else None,
        })
    return hits

def truncate_to_fit_model_window(
    question: str,
    hits: List[Dict[str, Any]],
    max_seq_len: int,
    max_new_tokens: int,
) -> Tuple[str, List[Dict[str, Any]]]:
    """
    Add chunks until the FULL chat prompt fits into (max_seq_len - max_new_tokens).
    Uses REAL token count from tokenizer.apply_chat_template.
    """
    used = []
    parts = []

    budget = max_seq_len - max_new_tokens
    if budget < 256:
        budget = max_seq_len - 128

    for h in hits:
        candidate_parts = parts + [h.get("text", "").strip()]
        candidate_context = "\n\n---\n\n".join([p for p in candidate_parts if p])

        user_content = (
            "Use the following course material excerpts to answer.\n"
            "Answer for the C language (not C++).\n\n"
            f"COURSE MATERIAL:\n{candidate_context}\n\n"
            f"QUESTION:\n{question}"
        )
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_content},
        ]

        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

        if ids.shape[1] <= budget:
            parts = candidate_parts
            used.append(h)
        else:
            break

    return "\n\n---\n\n".join([p for p in parts if p]), used

def build_rag_context(
    query: str,
    embedder: SentenceTransformer,
    persist_dir: str,
    top_k: int = TOP_K,
) -> Dict[str, Any]:
    hits = retrieve_chunks(query=query, embedder=embedder, persist_dir=persist_dir, top_k=top_k)

    context_text, used = truncate_to_fit_model_window(
        question=query,
        hits=hits,
        max_seq_len=MAX_SEQ_LEN,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
    )

    citations = []
    for i, h in enumerate(used, start=1):
        meta = h.get("meta", {}) or {}
        citations.append({
            "idx": i,
            "topic_id": meta.get("topic_id"),
            "chunk_id": meta.get("chunk_id"),
            "source_files": parse_list_field(meta.get("source_files", "")),
            "distance": h.get("distance"),
        })

    return {"context": context_text, "citations": citations}

# =========================
# 7) Build / load RAG index
# =========================
embedder = SentenceTransformer(EMBED_MODEL_NAME)

df_chunks = load_chunks_csv(CHUNK_CSV_PATH)

# If Chroma on Drive ever becomes "readonly" after crashes:
# import shutil
# if os.path.exists(CHROMA_DIR):
#     shutil.rmtree(CHROMA_DIR)

os.makedirs(CHROMA_DIR, exist_ok=True)
build_chroma_index_from_df(df_chunks, embedder, persist_dir=CHROMA_DIR, batch_size=128)

# =========================
# 8) Compare 3 variants
# =========================
def answer_base(q: str) -> str:
    return generate_answer(base_model, q, context="")

def answer_lora(q: str) -> str:
    return generate_answer(lora_model, q, context="")

def answer_lora_rag(q: str) -> Tuple[str, List[Dict[str, Any]]]:
    rag = build_rag_context(q, embedder, persist_dir=CHROMA_DIR, top_k=TOP_K)
    ans = generate_answer(lora_model, q, context=rag["context"])
    return ans, rag["citations"]

tests = [
    "In C, what is the difference between declaring and initializing a variable? Give an example.",
    "Write a simple for-loop in C that prints numbers from 1 to 5 and explain how it works.",
    "What is the purpose of a function prototype in C?",
]

for t in tests:
    print("\n" + "="*90)
    print("SUBJECT:", SUBJECT.upper())
    print("Q:", t)

    print("\nBASE:\n", answer_base(t))
    print("\nLoRA:\n", answer_lora(t))

    ans_rag, cites = answer_lora_rag(t)
    print("\nLoRA + RAG:\n", ans_rag)
    print("\nCITATIONS:")
    for c in cites:
        print(" ", c)
